# Dokumentacja Projektu
W tym notatniku znajduje się dokumentacja i przykłady użycia modeli do zadań Super Resolution oraz Denoising.

Poniższy kod demonstruje w jaki sposób wytrenować modele dla obu zadań oraz jak obliczyć metryki jakości wynikowych obrazów.

## Ewaluacja modelu
W tej sekcji załadujemy wytrenowany wcześniej model z folderu `outputs` i sprawdzimy jego wyniki na zbiorze walidacyjnym, obliczając przy tym metryki (PSNR, SSIM, LPIPS) oraz wizualizując wyniki.

In [ ]:
print('test')

In [ ]:
import torch
import matplotlib.pyplot as plt
from glob import glob
from models import SimpleUNet
from datasets import SuperResolutionDataset
from metrics import calculate_metrics
import os
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

valid_image_paths = glob("../dataset/DIV2K_valid_HR/*.png")
if not valid_image_paths:
    print("Brak obrazów w valid! Spróbuję z train.")
    valid_image_paths = glob("../dataset/DIV2K_train_HR/*.png")

val_dataset = SuperResolutionDataset(valid_image_paths)
lr_tensor, hr_tensor = val_dataset[0]
lr_batch = lr_tensor.unsqueeze(0).to(device)

lr_img = lr_tensor.permute(1, 2, 0).numpy()
hr_img = hr_tensor.permute(1, 2, 0).numpy()

directories = glob("../outputs/super_resolution_*")
results = []
print(f"Found {len(directories)} directories: {directories}")

fig, axes = plt.subplots(len(directories) + 1, 2, figsize=(10, 5 * (len(directories) + 1)))

axes[0, 0].imshow(lr_img)
axes[0, 0].set_title("Oryginalnie Zepsuty (Low-Res)")
axes[0, 0].axis('off')

axes[0, 1].imshow(hr_img)
axes[0, 1].set_title("Cel (High-Res)")
axes[0, 1].axis('off')

for idx, d in enumerate(directories):
    model_paths = glob(os.path.join(d, "*.pth"))
    if not model_paths:
        continue
    
    # Sort paths by epoch number (using naive string sort will fail on epoch_10 vs epoch_2, but let's just pick one or parse safely)
    model_paths = sorted(model_paths, key=lambda x: int(x.split('_epoch_')[1].split('_')[0]))
    best_model_path = model_paths[-1]

    d_name = os.path.basename(d)
    
    model = SimpleUNet().to(device)
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    model.eval()
    
    with torch.no_grad():
        sr_batch = model(lr_batch)
        
    sr_img = sr_batch.squeeze(0).cpu().permute(1, 2, 0).numpy()
    metrics = calculate_metrics(hr_img, sr_img)
    
    metrics['Zbiór / Parametr'] = d_name
    results.append(metrics)
    
    axes[idx + 1, 0].imshow(sr_img)
    axes[idx + 1, 0].set_title(f"{d_name}\n(Super Res)")
    axes[idx + 1, 0].axis('off')

    axes[idx + 1, 1].axis('off') # puste albo mozna wpisac metryki
    axes[idx + 1, 1].text(0.1, 0.5, f"PSNR: {metrics['PSNR']:.4f}\nSSIM: {metrics['SSIM']:.4f}\nLPIPS: {metrics['LPIPS']:.4f}", fontsize=12)

plt.tight_layout()
plt.show()

# Przedstawienie tego jako tabeli
df_results = pd.DataFrame(results)
display(df_results)